In [3]:
import os 
import pandas as pd
import numpy as np

In [6]:
import os
import pandas as pd
import numpy as np

def analyze_factors(data_dir: str):
    """
    Scans all CSVs in the cleaned data folder and reports:
    - Date range (start/end)
    - Frequency (daily/weekly/monthly/quarterly)
    - Row count
    - Columns
    - Missing value %
    """

    def detect_frequency(date_series):
        if len(date_series) < 2:
            return "unknown"
        diffs = date_series.sort_values().diff().dropna().dt.days
        median_diff = diffs.median()
        if median_diff <= 2:
            return "daily"
        elif median_diff <= 10:
            return "weekly"
        elif median_diff <= 35:
            return "monthly"
        elif median_diff <= 100:
            return "quarterly"
        else:
            return "yearly"

    def find_date_column(df):
        for col in df.columns:
            if any(kw in col.lower() for kw in ["date", "time", "period", "month", "year"]):
                try:
                    parsed = pd.to_datetime(df[col], infer_datetime_format=True, errors="coerce")
                    if parsed.notna().sum() > len(df) * 0.8:
                        return col, parsed
                except:
                    continue
        # fallback: try first column
        try:
            parsed = pd.to_datetime(df.iloc[:, 0], infer_datetime_format=True, errors="coerce")
            if parsed.notna().sum() > len(df) * 0.8:
                return df.columns[0], parsed
        except:
            pass
        return None, None

    results = []
    files = [f for f in os.listdir(data_dir) if f.endswith(".csv")]

    if not files:
        print("No CSV files found in directory.")
        return

    for file in sorted(files):
        path = os.path.join(data_dir, file)
        try:
            df = pd.read_csv(path)
            date_col, dates = find_date_column(df)

            if dates is not None and dates.notna().sum() > 1:
                freq    = detect_frequency(dates.dropna())
                start   = dates.min().date()
                end     = dates.max().date()
                n_years = round((dates.max() - dates.min()).days / 365, 1)
            else:
                freq = start = end = n_years = "no date col found"

            missing_pct = round(df.isnull().mean().mean() * 100, 2)
            non_date_cols = [c for c in df.columns if c != date_col]

            results.append({
                "file"         : file,
                "date_col"     : date_col,
                "start"        : start,
                "end"          : end,
                "years"        : n_years,
                "frequency"    : freq,
                "rows"         : len(df),
                "columns"      : non_date_cols,
                "missing_%"    : missing_pct,
            })

        except Exception as e:
            results.append({"file": file, "error": str(e)})

    # Print clean summary
    print(f"\n{'='*80}")
    print(f"FACTOR DATA AUDIT — {len(results)} files found")
    print(f"{'='*80}\n")

    for r in results:
        if "error" in r:
            print(f"❌ {r['file']} → ERROR: {r['error']}\n")
            continue
        print(f"📁 {r['file']}")
        print(f"   Date col   : {r['date_col']}")
        print(f"   Range      : {r['start']} → {r['end']} ({r['years']} years)")
        print(f"   Frequency  : {r['frequency']}")
        print(f"   Rows       : {r['rows']}")
        print(f"   Columns    : {r['columns']}")
        print(f"   Missing    : {r['missing_%']}%")
        print()

    return results


# ── Run it ──────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    DATA_DIR = "/Users/surisettivamsikrishna/Downloads/Vamsi Pc/Qoin/Cleaned data"
    results = analyze_factors(DATA_DIR)


FACTOR DATA AUDIT — 14 files found

📁 ICICI_Bank_Quarterly_Results_FY2010_FY2026 copy.csv
   Date col   : None
   Range      : no date col found → no date col found (no date col found years)
   Frequency  : no date col found
   Rows       : 68
   Columns    : ['Fiscal Year', 'Quarter', 'Period', 'NII (₹ Cr)', 'NII QoQ Change (₹ Cr)', 'NII YoY Change (₹ Cr)', 'Net Profit (₹ Cr)', 'PAT QoQ Change (₹ Cr)', 'PAT YoY Change (₹ Cr)', 'Data Source']
   Missing    : 1.47%

📁 India 10-Year Bond Yield Historical Data.csv
   Date col   : Date
   Range      : 2010-01-01 → 2026-05-13 (16.4 years)
   Frequency  : daily
   Rows       : 4072
   Columns    : ['Price', 'Open', 'High', 'Low', 'Change %']
   Missing    : 0.0%

📁 India 5-Year Bond Yield Historical Data.csv
   Date col   : Date
   Range      : 2010-01-01 → 2026-05-13 (16.4 years)
   Frequency  : daily
   Rows       : 4066
   Columns    : ['Price', 'Open', 'High', 'Low', 'Change %']
   Missing    : 0.0%

📁 bank_nifty_daily.csv
   Date col  

/var/folders/10/4p6gdwfx3g74kjfj2_xqxz140000gn/T/ipykernel_8318/789148564.py:35: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  parsed = pd.to_datetime(df[col], infer_datetime_format=True, errors="coerce")
/var/folders/10/4p6gdwfx3g74kjfj2_xqxz140000gn/T/ipykernel_8318/789148564.py:35: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[col], infer_datetime_format=True, errors="coerce")
/var/folders/10/4p6gdwfx3g74kjfj2_xqxz140000gn/T/ipykernel_8318/789148564.py:35: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, se

In [8]:
fidi = pd.read_csv('/Users/surisettivamsikrishna/Downloads/Fii Dii Trading activity.csv')

In [10]:
fidi.dtypes

Date                       object
FII_Gross_Purchase        float64
FII_Gross_Sales           float64
FII_Net_Purchase/Sales    float64
DII_Gross_Purchase        float64
DII_Gross_Sales           float64
DII_Net_Purchase/Sales    float64
dtype: object

In [11]:
import pandas as pd

# Load the file
fidi = pd.read_csv('/Users/surisettivamsikrishna/Downloads/Fii Dii Trading activity.csv')

# 1. Basic Info
print(fidi.shape)
print(fidi.columns.tolist())
print(fidi.head())

# 2. Clean the data
fidi['Date'] = pd.to_datetime(fidi['Date'], errors='coerce')   # Convert Date to proper format
fidi = fidi.sort_values(by='Date')                              # Sort by date (oldest to newest)

# 3. Filter for 2010 to 2018
fidi_2010_2018 = fidi[(fidi['Date'] >= '2010-01-01') & (fidi['Date'] <= '2018-12-31')]

# 4. Keep only important columns (adjust if needed)
cols = ['Date', 'FII_Net_Purchase/Sales', 'DII_Net_Purchase/Sales']   # Most common useful ones
fidi_clean = fidi_2010_2018[cols].copy()

print(fidi_clean.head())
print(fidi_clean.tail())

(3770, 7)
['Date', 'FII_Gross_Purchase', 'FII_Gross_Sales', 'FII_Net_Purchase/Sales', 'DII_Gross_Purchase', 'DII_Gross_Sales', 'DII_Net_Purchase/Sales']
         Date  FII_Gross_Purchase  FII_Gross_Sales  FII_Net_Purchase/Sales  \
0  01-01-2008              888.69          1350.10                 -461.41   
1  01-01-2009              260.28           168.24                   92.04   
2  01-01-2013              885.51           220.46                  665.05   
3  01-01-2014              225.12           214.96                   10.16   
4  01-01-2015              248.38           230.18                   18.20   

   DII_Gross_Purchase  DII_Gross_Sales  DII_Net_Purchase/Sales  
0             1284.38           977.44                  306.94  
1              598.23           437.82                  160.41  
2              857.63          1263.77                 -406.14  
3              359.22           581.34                 -222.12  
4              691.07           671.51               

In [12]:
fidi.head()

,Date,FII_Gross_Purchase,FII_Gross_Sales,FII_Net_Purchase/Sales,DII_Gross_Purchase,DII_Gross_Sales,DII_Net_Purchase/Sales
144,2007-04-02,2010.80,2520.13,-509.33,0.0,0.0,0.0
256,2007-04-03,2179.39,2185.40,-6.01,0.0,0.0,0.0
386,2007-04-04,2126.12,2171.17,-45.05,0.0,0.0,0.0
513,2007-04-05,1632.16,1717.96,-85.80,0.0,0.0,0.0
1008,2007-04-09,1423.75,930.39,493.36,0.0,0.0,0.0


In [13]:
fii= pd.read_csv('/Users/surisettivamsikrishna/Downloads/Vamsi Pc/Qoin/Cleaned data/fii_dii_2018_to_2026.csv') 

In [16]:
fidi.dtypes

Date                      datetime64[ns]
FII_Gross_Purchase               float64
FII_Gross_Sales                  float64
FII_Net_Purchase/Sales           float64
DII_Gross_Purchase               float64
DII_Gross_Sales                  float64
DII_Net_Purchase/Sales           float64
dtype: object

In [20]:
import pandas as pd

# ================== LOAD BOTH FILES ==================
# Your Kaggle file (older data)
old = pd.read_csv('/Users/surisettivamsikrishna/Downloads/Fii Dii Trading activity.csv')   # change path if needed

# Your new file (2018 onwards)
new = pd.read_csv('/Users/surisettivamsikrishna/Downloads/Vamsi Pc/Qoin/Cleaned data/fii_dii_2018_to_2026.csv')   # ← Update this path

# ================== STANDARDIZE COLUMN NAMES ==================
old = old.rename(columns={
    'FII_Gross_Purchase': 'FII_Buy',
    'FII_Gross_Sales': 'FII_Sell',
    'FII_Net_Purchase/Sales': 'FII_Net',
    'DII_Gross_Purchase': 'DII_Buy',
    'DII_Gross_Sales': 'DII_Sell',
    'DII_Net_Purchase/Sales': 'DII_Net'
})

# Convert dates
old['Date'] = pd.to_datetime(old['Date'], dayfirst=True, errors='coerce')
new['Date'] = pd.to_datetime(new['Date'], errors='coerce')

# Keep only useful columns from new file
new = new[['Date', 'FII_Buy', 'FII_Sell', 'FII_Net', 'DII_Buy', 'DII_Sell', 'DII_Net', 
           'Nifty_Close', 'Nifty_Change', 'Nifty_%Chg']]

# ================== MERGE ==================
# Combine old (2010-2017) + new (2018+)
combined = pd.concat([old, new], ignore_index=True)

# Remove duplicates + sort by date
combined = combined.drop_duplicates(subset=['Date'])
combined = combined.sort_values('Date').reset_index(drop=True)

# Filter from 2010 onwards
final = combined[combined['Date'] >= '2010-01-01'].copy()

print("Final Shape:", final.shape)
print(final.head())
print(final.tail())

# Save the big file
final.to_csv('/Users/surisettivamsikrishna/Downloads/Vamsi Pc/Qoin/Cleaned data/updatedfiidii.csv', index=False)

Final Shape: (3095, 10)
          Date  FII_Buy  FII_Sell  FII_Net  DII_Buy  DII_Sell  DII_Net  \
675 2010-01-02  2035.96   2530.62  -494.66  1130.61    930.78   199.83   
676 2010-01-04  2405.62   2299.22   106.40  1665.25   1212.92   452.33   
677 2010-01-06  1801.93   2328.42  -526.49  1220.02   1009.40   210.62   
678 2010-01-07  1549.36   1709.97  -160.61  1153.52   1160.22    -6.70   
679 2010-01-09  2683.31   2323.50   359.81  1210.05   1041.49   168.56   

     Nifty_Close  Nifty_Change  Nifty_%Chg  
675          NaN           NaN         NaN  
676          NaN           NaN         NaN  
677          NaN           NaN         NaN  
678          NaN           NaN         NaN  
679          NaN           NaN         NaN  
           Date  FII_Buy  FII_Sell  FII_Net  DII_Buy  DII_Sell  DII_Net  \
3765 2022-12-01  6981.26   7982.83 -1001.57  7458.25   6126.24  1332.01   
3766 2022-12-04  7047.27  10175.66 -3128.39  6629.33   5759.32   870.01   
3767 2022-12-05  6018.03  11273.78 -

In [21]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

# ── Step 1: Your factor matrix ────────────────────────────────
# Assume df is your cleaned, forward-filled, differenced dataframe
# Columns: Gold, Crude, USDINR, VIX, FII, DII, Bonds, US10Y,
#          TradeDeficit, GDP, CPI, RepoRate, Nifty, SectorIdx, Volume
#          + ICICI_close as target

factor_cols = [
    'Gold', 'Crude', 'USDINR', 'VIX',
    'FII', 'DII', 'IndBonds', 'US10Y',
    'd_TradeDeficit', 'd_GDP', 'd_CPI', 'd_RepoRate',
    'Nifty', 'SectorIdx', 'Volume'
]

X = df[factor_cols].values          # shape: (n_days, 14)
y = df['ICICI_close'].values        # shape: (n_days,)

# ── Step 2: Standardise — mandatory before projection ─────────
# Each factor is on a different scale (Gold in ₹, VIX in %, etc.)
# Without this, Gold swamps everything just because it's large numbers
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # now every column: mean=0, std=1

# ── Step 3: Project ICICI price onto the factor space ─────────
# This fits: Price = β₀ + β₁F₁ + β₂F₂ + ... + β₁₄F₁₄
reg = LinearRegression()
reg.fit(X_scaled, y)

price_projected = reg.predict(X_scaled)  # the linear combination

# ── Step 4: The residual — this is the KEY output ─────────────
residual = y - price_projected

# Residual = what the factors CANNOT explain
# If residual > 0  → price is above what factors justify  → overpriced
# If residual < 0  → price is below what factors justify  → underpriced

# ── Step 5: See how much each factor contributes ──────────────
loadings = pd.Series(reg.coef_, index=factor_cols).sort_values()
print("Factor loadings (standardised):")
print(loadings)
# Positive loading = factor going up → price goes up
# Negative loading = factor going up → price goes down

# ── Step 6: Check how well the factors explain price ──────────
r2 = reg.score(X_scaled, y)
print(f"\nR² = {r2:.3f}")
# R² = 0.7 means 70% of price movement explained by your 14 factors
# Residual holds the remaining 30% — that's your mean-reversion target

# ── Step 7: Plot actual vs projected ──────────────────────────
plt.figure(figsize=(14, 5))
plt.subplot(2, 1, 1)
plt.plot(df.index, y, label='Actual ICICI', alpha=0.8)
plt.plot(df.index, price_projected, label='Factor projection', alpha=0.8)
plt.legend()
plt.title('Actual vs factor-projected price')

plt.subplot(2, 1, 2)
plt.plot(df.index, residual, color='red', alpha=0.7)
plt.axhline(0, color='black', linewidth=0.5)
plt.title('Residual (what OU will model)')
plt.tight_layout()
plt.show()

TypeError: 'module' object is not subscriptable

In [22]:
print(type(df))
print(df)

<class 'module'>
<module 'pandas' from '/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/pandas/__init__.py'>


In [4]:
import pandas as pd
dd= pd.read_csv("/Users/surisettivamsikrishna/Downloads/Vamsi Pc/Qoin/Model data/master_raw_aligned.csv")
dd.head()

,Unnamed: 0,ICICI_Close,Nifty_Close,BankNifty_Close,Gold_Close,USD_INR,Bond_10Y,Bond_5Y,US_10Y,US_5Y,FII_Net,DII_Net,CPI_Index,WPI_Index,Trade_Balance_Billion_USD,Repo_Rate_%,CRR_%,Real_GDP_YoY_Growth_%,Net Profit (₹ Cr),NII (₹ Cr)
0,2010-01-04,129.260223,5094.149902,8919.196289,1089.199951,46.27,7.676,7.253,3.841,2.652,325.91,-1300.31,65.03028,78.463211,-12.47613,6.0,5.0,10.149065,878.0,2100.0
1,2010-01-05,129.260223,5094.149902,8919.196289,1089.199951,46.13,7.701,7.211,3.755,2.558,325.91,-1300.31,65.03028,78.463211,-12.47613,6.0,5.0,10.149065,878.0,2100.0
2,2010-01-06,129.260223,5094.149902,8919.196289,1089.199951,45.72,7.734,7.241,3.808,2.573,325.91,-1300.31,65.03028,78.463211,-12.47613,6.0,5.0,10.149065,878.0,2100.0
3,2010-01-07,129.260223,5094.149902,8919.196289,1089.199951,45.67,7.778,7.266,3.822,2.600,325.91,-1300.31,65.03028,78.463211,-12.47613,6.0,5.0,10.149065,878.0,2100.0
4,2010-01-08,129.260223,5094.149902,8919.196289,1089.199951,45.50,7.773,7.266,3.808,2.566,325.91,-1300.31,65.03028,78.463211,-12.47613,6.0,5.0,10.149065,878.0,2100.0
